In [1]:
import pandas as pd

text = pd.read_csv ('/Users/arjunbubbar/Desktop/Jetlearn/Data Science/Datasets/sentiments.txt', sep=';',names = ['sentence','sentiment'])

print (text.info ())
print (text ['sentiment'].value_counts ())

text ['sentiment'] = text ['sentiment'].replace ({'joy':1,'sadness':0,'anger':0,'fear':0,'love':1,'surprise':1})

print (text ['sentiment'].value_counts ())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16000 entries, 0 to 15999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   sentence   16000 non-null  object
 1   sentiment  16000 non-null  object
dtypes: object(2)
memory usage: 250.1+ KB
None
sentiment
joy         5362
sadness     4666
anger       2159
fear        1937
love        1304
surprise     572
Name: count, dtype: int64
sentiment
0    8762
1    7238
Name: count, dtype: int64


/var/folders/tw/tb35p4tn71s66v714xyr3zcr0000gn/T/ipykernel_11957/277699847.py:8: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  text ['sentiment'] = text ['sentiment'].replace ({'joy':1,'sadness':0,'anger':0,'fear':0,'love':1,'surprise':1})


In [2]:
# Regular expression - i want to see some text followed by .py, anything then .py 
# Create sequences of characters so that it can match it with text
# nltk is nlp
# wordnet is a database of words, corpus
# need to download the lemmatiser

import re
import nltk 

nltk.download ('stopwords')
nltk.download ('wordnet')
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer



[nltk_data] Error loading stopwords: <urlopen error [Errno 8] nodename
[nltk_data]     nor servname provided, or not known>
[nltk_data] Error loading wordnet: <urlopen error [Errno 8] nodename
[nltk_data]     nor servname provided, or not known>


In [3]:
# anything which is not a capital a-z or small, is subbed to a space in regular expression, take the sentence, ^ = not
# removes all special characters, lower is for lower case
# word tokenisation, split function identifies a space
# join the elements of lemma list by putting space in between the words

wnl = WordNetLemmatizer ()

def textprocessing (txt):
    transformedtext = []
    stop = stopwords.words ('english')
    for sentence in txt:
        item = re.sub ('[^a-zA-Z]',' ',sentence)
        item = item.lower ()
        wordlist = item.split ()
        lemmalist = []
        for word in wordlist:
            if word not in stop:
                lemmalist.append (wnl.lemmatize (word))
        transformedtext.append (' '.join (lemmalist))
    return transformedtext



                
                












In [ ]:
transformedtext = textprocessing (text ['sentence'])

print (text ['sentence'].iloc [500:505])
print (transformedtext [500:505])


500    i love children s literature authors who don t...
501    i was soo quiet it was a mixture of not sleepi...
502    i do feel that they are greedy and money hungr...
503     i feel so fucked up now i want to shut myself up
504    i feel very passionate about a certain topic i...
Name: sentence, dtype: object
['love child literature author feel need dumb thing kid', 'soo quiet mixture sleeping well feeling bit isolated big group', 'feel greedy money hungry absolutely', 'feel fucked want shut', 'feel passionate certain topic love backing position actual knowledge fact instead relying solely opinion']


In [ ]:
# count vectoriser, each unique word becomes column in matrix 
# converts text into numericals like the tfidf one, in movie descriptions
# ngram range - 1,1 single words e.g. student passionate
# 2,2 feature would be 'i feel' 'very nice' - so 1,2 gets both version
# features assigned numbers alphabetically

from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer (ngram_range= (1,2))

features = cv.fit_transform (transformedtext)

print (features.shape)

print (features [500])



(16000, 106712)
<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 17 stored elements and shape (1, 106712)>
  Coords	Values
  (0, 29986)	1
  (0, 13682)	1
  (0, 92887)	1
  (0, 56547)	1
  (0, 62892)	1
  (0, 30876)	1
  (0, 49801)	1
  (0, 54799)	1
  (0, 5907)	1
  (0, 23990)	1
  (0, 56587)	1
  (0, 13741)	1
  (0, 54800)	1
  (0, 5912)	1
  (0, 62968)	1
  (0, 24026)	1
  (0, 93044)	1


In [9]:
from sklearn.model_selection import train_test_split

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import classification_report, f1_score, confusion_matrix


xtrain, xtest, ytrain, ytest = train_test_split (features, text ['sentiment'])

rfc = RandomForestClassifier ()

rfc.fit (xtrain,ytrain)

predy = rfc.predict (xtest)

print (predy)
print (confusion_matrix (ytest,predy))
print (classification_report (ytest,predy))
print (f1_score (ytest,predy))




[0 0 0 ... 0 0 0]
[[2149   72]
 [ 132 1647]]
              precision    recall  f1-score   support

           0       0.94      0.97      0.95      2221
           1       0.96      0.93      0.94      1779

    accuracy                           0.95      4000
   macro avg       0.95      0.95      0.95      4000
weighted avg       0.95      0.95      0.95      4000

0.9416809605488851


In [10]:
def pred (sentence):
    result = textprocessing (sentence)
    vectorisedtext = cv.transform (result)
    prediction = rfc.predict (vectorisedtext)
    print (prediction)
    

In [12]:
# need to put it into a list, defined the function

pred (['I love football'])

[0]
